# SongFormer: Music Structure Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SidSaxena/SongFormer/blob/cross-platform/notebooks/SongFormer-Colab.ipynb)

**SongFormer** is a music structure analysis framework that segments songs into labeled sections (intro, verse, chorus, bridge, etc.).

This notebook provides two options:
- **Option A:** Run the Gradio web app (interactive, upload audio and see results)
- **Option B:** Run batch inference (process multiple audio files at once)

**Requirements:** A GPU runtime is recommended. Go to `Runtime → Change runtime type → GPU`.

Paper: [arXiv:2510.02797](https://arxiv.org/abs/2510.02797) | GitHub: [ASLP-lab/SongFormer](https://github.com/ASLP-lab/SongFormer)

## 1. Check GPU

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"GPU Memory: {mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime → Change runtime type → GPU")

## 2. Clone repository and install dependencies

In [ ]:
!git clone https://github.com/SidSaxena/SongFormer.git -b cross-platform
%cd SongFormer
!git submodule update --init --recursive

In [ ]:
!pip install torch==2.8.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -r requirements.txt

## 3. Download model checkpoints

Checkpoints are **~1.33 GB** total:
- `pretrained_msd.pt` (MusicFM) — 1.23 GB
- `SongFormer.safetensors` — 99.6 MB
- `msd_stats.json` — 2.2 KB

**Choose storage option:**
- `STORE_ON_DRIVE = True` — Saves checkpoints to Google Drive. Downloaded once, persists across sessions. Uses ~1.33 GB of Drive space.
- `STORE_ON_DRIVE = False` — Downloads fresh each session. No Drive space used, but re-downloads every time.

In [ ]:
#@markdown **Set to True to persist checkpoints on Google Drive (uses ~1.33 GB)**
STORE_ON_DRIVE = False  #@param {type:"boolean"}

import os
import shutil

REPO_CKPTS = '/content/SongFormer/src/SongFormer/ckpts'

if STORE_ON_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CKPTS = '/content/drive/MyDrive/SongFormer/ckpts'
    os.makedirs(DRIVE_CKPTS, exist_ok=True)

    # Symlink Drive checkpoints into the repo
    if os.path.isdir(REPO_CKPTS) and not os.path.islink(REPO_CKPTS):
        for f in os.listdir(REPO_CKPTS):
            src = os.path.join(REPO_CKPTS, f)
            dst = os.path.join(DRIVE_CKPTS, f)
            if not os.path.exists(dst):
                if os.path.isdir(src):
                    shutil.copytree(src, dst)
                else:
                    shutil.copy2(src, dst)
        shutil.rmtree(REPO_CKPTS)
    elif not os.path.exists(REPO_CKPTS):
        pass

    if not os.path.islink(REPO_CKPTS):
        os.symlink(DRIVE_CKPTS, REPO_CKPTS)
    print(f"Checkpoints will be stored at: {DRIVE_CKPTS}")
else:
    print("Checkpoints will be downloaded to the session (not persisted).")

# Download any missing checkpoints
os.chdir('/content/SongFormer/src/SongFormer')
from utils.fetch_pretrained import download_all
download_all()
os.chdir('/content/SongFormer')

# Verify
print("\nDownloaded checkpoints:")
for f in ['MusicFM/msd_stats.json', 'MusicFM/pretrained_msd.pt', 'SongFormer.safetensors']:
    path = os.path.join(REPO_CKPTS, f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")

---
## Option A: Gradio Web App

Run the interactive Gradio app. Colab will generate a public URL you can open in your browser.

**How to use:**
1. Run the cell below
2. Click the public URL that appears (e.g., `https://xxxxx.gradio.live`)
3. Upload an audio file and click "Analyze Music Structure"

In [ ]:
%cd /content/SongFormer
!python app.py

---
## Option B: Batch Inference

Process multiple audio files at once using the command-line inference script.

**How to use:**
1. Upload your audio files (see cell below)
2. Create an `.scp` file listing the paths
3. Run inference
4. Download results

### B1. Upload audio files

Upload your audio files using the cell below, or copy them from Google Drive.

In [ ]:
import os

# Create a directory for your audio files
AUDIO_DIR = '/content/audio'
os.makedirs(AUDIO_DIR, exist_ok=True)

# Option 1: Upload files from your computer
from google.colab import files
print("Upload audio files (mp3, wav, flac, etc.):")
uploaded = files.upload()
for filename, data in uploaded.items():
    dst = os.path.join(AUDIO_DIR, filename)
    with open(dst, 'wb') as f:
        f.write(data)
    print(f"  Saved: {dst}")

# Option 2: Copy from Google Drive (uncomment and edit the path)
# import shutil
# drive_audio_dir = '/content/drive/MyDrive/my_audio_files'
# for f in os.listdir(drive_audio_dir):
#     shutil.copy2(os.path.join(drive_audio_dir, f), AUDIO_DIR)

audio_files = [os.path.join(AUDIO_DIR, f) for f in sorted(os.listdir(AUDIO_DIR))]
print(f"\nTotal files: {len(audio_files)}")

### B2. Create `.scp` file and run inference

In [ ]:
import os

# Create .scp file (one audio path per line)
AUDIO_DIR = '/content/audio'
SCP_PATH = '/content/audio_list.scp'
OUTPUT_DIR = '/content/SongFormer/output'

audio_files = sorted([
    os.path.join(AUDIO_DIR, f)
    for f in os.listdir(AUDIO_DIR)
    if f.endswith(('.mp3', '.wav', '.flac', '.ogg', '.m4a'))
])

with open(SCP_PATH, 'w') as f:
    for path in audio_files:
        f.write(path + '\n')

print(f"Created {SCP_PATH} with {len(audio_files)} files")
print("Files:")
for p in audio_files:
    print(f"  {p}")

In [ ]:
%cd /content/SongFormer/src/SongFormer

!python infer/infer.py \
    --input_path /content/audio_list.scp \
    --output_path /content/SongFormer/output \
    --model SongFormer \
    --checkpoint SongFormer.safetensors \
    --config_path SongFormer.yaml \
    --gpu_num 1

print("\n=== Results ===")
import json
for f in sorted(os.listdir('/content/SongFormer/output')):
    if f.endswith('.json'):
        print(f"\n{f}:")
        with open(f'/content/SongFormer/output/{f}') as fh:
            data = json.load(fh)
            for seg in data:
                print(f"  {seg['start']:.1f}s - {seg['end']:.1f}s: {seg['label']}")

### B3. Download results

In [ ]:
import shutil
from google.colab import files

OUTPUT_DIR = '/content/SongFormer/output'
ZIP_PATH = '/content/songformer_results.zip'

shutil.make_archive(ZIP_PATH.replace('.zip', ''), 'zip', OUTPUT_DIR)
print(f"Zipped results to: {ZIP_PATH}")

# Download the zip file
files.download(ZIP_PATH)